# Fipiran Update

In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import re
import os
import ast  # For safely converting string representations of lists to actual lists
import jdatetime  # Library for converting Gregorian to Jalali
import openpyxl
import numpy as np



# Function to fetch data for a specific date from the API
def fetch_data(date):
    print(f"Fetching data from API for date: {date}")
    url = f"https://fund.fipiran.ir/api/v1/fund/fundcompare?date={date}"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        print(f"Data fetched successfully for {date}")
        return response.json()
    else:
        print(f"Failed to fetch data for {date}. Status code: {response.status_code}")
        return None
    
    
# Function to fetch fund types
def fetch_fund_types():
    url = "https://fund.fipiran.ir/api/v1/fund/fundtype"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        items_df = pd.json_normalize(data['items'])
        items_df['name'] = items_df['name'].str.replace(r'^در\s*', '', regex=True)
        print("Fund types fetched successfully.")
        return items_df
    else:
        print(f"Failed to fetch fund types. Status code: {response.status_code}")
        return pd.DataFrame()


# Function to fetch full data for given regNos and date range
def fetch_instrument_data(regno, date, session):
    print(f"Fetching full instrument data for regNo: {regno} on {date}")
    url = f"https://fund.fipiran.ir/api/v1/fund/getfund?regno={regno}&date={date}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = session.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()

        if data and "item" in data:
            item_data = data["item"]
            df = pd.DataFrame([item_data])
            df.drop(columns=['name'], errors='ignore', inplace=True)
            return df
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for regNo: {regno} on {date} - {e}")
    return None


# Function to save JSON data to a DataFrame (now extracting from item column)
def save_to_dataframe(data):
    if data and 'items' in data:
        df = pd.DataFrame(data['items'])  # Flatten the nested 'items' list
        return df
    else:
        return pd.DataFrame()
    

# Function to process and flatten the mutualFundLicenses column into multiple columns
def process_mutual_fund_licenses(file_path):
    # Load the file into a DataFrame
    df = pd.read_excel(file_path, engine='openpyxl')

    # Check if the 'mutualFundLicenses' column exists in the DataFrame
    if 'mutualFundLicenses' in df.columns:
        # Convert the string representations of lists into actual lists
        df['mutualFundLicenses'] = df['mutualFundLicenses'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

        # Extract the dictionary from the list in the 'mutualFundLicenses' column
        df['mutualFundLicenses'] = df['mutualFundLicenses'].apply(
            lambda x: x[0] if isinstance(x, list) and len(x) > 0 else {}  # Extract first dictionary in the list
        )

        # Normalize the dictionary into separate columns
        licenses_df = pd.json_normalize(df['mutualFundLicenses'])

        # Drop the 'regNo' column from licenses_df to avoid the column overlap
        licenses_df = licenses_df.drop(columns=['regNo'], errors='ignore')

        # Merge the normalized data back with 'regNo' and drop the original 'mutualFundLicenses' column
        result_df = df.drop(columns=['mutualFundLicenses']).join(licenses_df)

        # Save the result to a new file or the same file with the sheet name changed to 'Fund_data'
        result_df.to_excel(file_path, index=False, engine='openpyxl')
        print(f"Data successfully processed and saved to {file_path} with sheet name 'Fund_data'.")
        return result_df



# Function to fetch and process data for a date range
def fetch_and_process_data(start_date, end_date, csv_filename):
    fund_types_df = fetch_fund_types()
    all_data = []

    current_date = start_date
    while current_date <= end_date:
        date_str = current_date.strftime("%Y-%m-%d")
        data = fetch_data(date_str)
        if data:
            new_data_df = save_to_dataframe(data)
            if not new_data_df.empty:
                new_data_df['date'] = date_str
                all_data.append(new_data_df)
        current_date += timedelta(days=1)  # Move to next date

    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        if not fund_types_df.empty:
            combined_df = pd.merge(
                combined_df, fund_types_df, on='fundType', how='left'
            )

        # Save to CSV or Excel
        combined_df.to_csv(csv_filename, index=False)
        print(f"Data saved to {csv_filename}.")
        return combined_df
    else:
        print("No data fetched.")
        return pd.DataFrame()

def post_process_data(updated_csv_filename):
    try:
        print(f"Reading post-processing data from {updated_csv_filename}...")
        
        # Read as an Excel file instead of CSV
        df = pd.read_excel(updated_csv_filename, engine='openpyxl')

        # Normalize text to remove non-standard spaces and characters
        df = df.applymap(lambda x: re.sub(r'[\u200c\u00A0ـ]', ' ', str(x)) if isinstance(x, str) else x)

        # Convert column names to strings before renaming
        df.columns = df.columns.astype(str)

        # 1. Rename 'name_x' to 'fund_name'
        df.rename(columns={'name_x': 'fund_name'}, inplace=True)

        # 2. Remove ranking columns
        columns_to_remove = [
            'rankOf12Month', 'rankOf24Month', 'rankOf36Month',
            'rankOf48Month', 'rankOf60Month', 'rankLastUpdate'
        ]
        df.drop(columns=columns_to_remove, errors='ignore', inplace=True)

        # 3. Rename columns ending with 'Efficiency' to 'Return'
        df.rename(columns=lambda x: x.replace('Efficiency', 'Return') if 'Efficiency' in x else x, inplace=True)

        # 4. Rename 'efficiency' column to 'CumulativeReturn'
        df.rename(columns={'efficiency': 'CumulativeReturn'}, inplace=True)

        # 5. Clean 'websiteAddress' column
        if 'websiteAddress' in df.columns:
            df['websiteAddress'] = df['websiteAddress'].str.strip("[]'\"")

        # 6. Rename 'name_y' to 'fundType_name' after the merge step
        df.rename(columns={'name_y': 'fundType_name'}, inplace=True)

        # 7. Clean 'auditor' column
        if 'auditor' in df.columns:
            df['auditor'] = df['auditor'].apply(lambda x: re.sub(r'حسابرس\s*\((.*?)\)', r'\1', str(x)))

        # 8. Clean 'manager' column
        if 'manager' in df.columns:
            def clean_manager(x):
                x = str(x)
                x = re.sub(r'مدیر\s*\((.*?)\)', r'\1', x)
                x = re.sub(r'تامین سرمایه تمدن\s*\(سهامی عام\)', 'تامین سرمایه تمدن', x)
                x = re.sub(r'کارگزاری بانک ملی ایران\s*،\s*مدیر', 'کارگزاری بانک ملی ایران', x)
                return x
            df['manager'] = df['manager'].apply(clean_manager)

        # 9. Clean 'custodian' column
        if 'custodian' in df.columns:
            df['custodian'] = df['custodian'].apply(lambda x: re.sub(r'متولی\s*\((.*?)\)', r'\1', str(x)))

        # 10. Standardize specific names ("نو ویرا" -> "نوویرا")
        df = df.applymap(lambda x: re.sub(r'نو\s+ویرا', 'نوویرا', str(x)) if isinstance(x, str) else x)

        # 11. Convert scientific notation to normal numbers in fundSize, netAsset, and insCode columns
        for col in ['fundSize', 'netAsset', 'insCode']:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: f'{x:.0f}' if isinstance(x, (int, float)) else x)

        # 12. Standardize "سبد گردان" to "سبدگردان"
        df = df.applymap(lambda x: re.sub(r'سبد\s+گردان', 'سبدگردان', str(x)) if isinstance(x, str) else x)

        # 13. Normalize and standardize "سرمایه گذاری" to "سرمایه‌گذاری"
        df = df.applymap(lambda x: re.sub(r'سرمایه\s+گذاری', 'سرمایه‌گذاری', str(x)) if isinstance(x, str) else x)

        # 14. Fix missing spaces after "مشاور"
        df = df.applymap(lambda x: re.sub(r'(مشاور)(سرمایه‌گذاری)', r'\1 \2', str(x)) if isinstance(x, str) else x)

        # # 15. **Divide specific columns by 100**
        # percentage_columns = [
        #     'dailyReturn', 'weeklyReturn', 'monthlyReturn', 
        #     'quarterlyReturn', 'sixMonthReturn', 'annualReturn', 'CumulativeReturn', 'fiveBest',
        #     'stock', 'bond', 'other', 'cash', 'deposit', 'insInvPercent', 'legalPercent', 'naturalPercent',
        #     'retInvPercent',  
        # ]
        
        # for col in percentage_columns:
        #     if col in df.columns:
        #         df[col] = df[col] / 100

        # 16. **Format date columns properly**
        date_columns = ['initiationDate', 'date', 'lastModificationTime', 'registerDate', 'seoRegisterDate', 'startDate']

        # 17. **Convert "date" column to Jalali (Shamsi) and add "shamsi_date" column**
        if 'date' in df.columns:
            def gregorian_to_jalali(g_date):
                if pd.notna(g_date):  # Ensure it's a valid date
                    try:
                        g_date = pd.to_datetime(g_date)  # Ensure it's a datetime object
                        j_date = jdatetime.datetime.fromgregorian(year=g_date.year, month=g_date.month, day=g_date.day)
                        return j_date.strftime('%Y - %m - %d')  # Format as YYYY - MM - DD
                    except:
                        return None
                return None

            df['shamsi_date'] = df['date'].apply(gregorian_to_jalali)

        # 18. **Remove (سهامی خاص) and (سهامی عام) from names**
        df = df.applymap(lambda x: re.sub(r'\s*\(سهامی خاص\)', '', str(x)) if isinstance(x, str) else x)
        df = df.applymap(lambda x: re.sub(r'\s*\(سهامی عام \)\s*', '', str(x)) if isinstance(x, str) else x)

        # 19. **Remove سهامی عام when it appears separately at the end**
        df = df.applymap(lambda x: re.sub(r'\s*سهامی عام$', '', str(x)) if isinstance(x, str) else x)

        # 20. **Replace "نامشخصنامشخص" with an empty string**
        df = df.applymap(lambda x: '' if isinstance(x, str) and 'نامشخصنامشخص' in x else x)

        # Save updated data
        df.to_excel(updated_csv_filename, index=False, sheet_name='Fund_data', engine='openpyxl')
        print(f"Post-processing completed. Updated data saved to {updated_csv_filename}.")

    except FileNotFoundError:
        print(f"Error: {updated_csv_filename} does not exist. Please check the previous steps.")



#import numpy as np  # Add this import

def merge_instrument_data(regnos, dates, output_filename, csv_filename, updated_csv_filename):
    print(f"Starting to fetch full instrument data for {len(regnos)} regNos and {len(dates)} dates...")
    all_results = []
    
    with requests.Session() as session:
        for regno in regnos:
            for date in dates:
                result = fetch_instrument_data(regno, date, session)
                if result is not None:
                    all_results.append(result)

    if all_results:
        df_instruments = pd.concat(all_results, ignore_index=True)
        df_instruments.to_excel(output_filename, sheet_name='instruments', index=False)
        print(f"Instrument data saved to {output_filename}.")

        # Read fund data
        df_fund = pd.read_csv(csv_filename)

        # Read instrument data
        df_instruments = pd.read_excel(output_filename, sheet_name='instruments')

        # Convert 'regNo' safely in fund data
        df_fund['regNo'] = pd.to_numeric(df_fund['regNo'], errors='coerce').fillna(0).astype('int64')

        # Convert 'regNo' safely in instrument data
        df_instruments['regNo'] = pd.to_numeric(df_instruments['regNo'], errors='coerce').fillna(0).astype('int64')

        # Convert date columns to datetime
        df_fund['date'] = pd.to_datetime(df_fund['date']).dt.strftime('%Y-%m-%d')
        df_instruments['date'] = pd.to_datetime(df_instruments['date']).dt.strftime('%Y-%m-%d')

        # Drop columns with all NaN values in the instrument data
        df_instruments = df_instruments.dropna(axis=1, how="all")

        # Merge datasets
        df_final = df_fund.merge(df_instruments, on=['regNo', 'date'], how='left', suffixes=("_fund", "_instrument"))

        # Keep:
        # - Columns that end with '_instrument'.
        # - Columns that do NOT end with '_fund' or '_instrument' (such as 'regNo', 'date', etc.).
        df_final = df_final.loc[:, 
            df_final.columns.str.endswith("_fund") | ~df_final.columns.str.endswith(("_instrument", "_fund"))
        ]


        # # Remove '_instrument' suffix from column names
        # df_final.columns = df_final.columns.str.replace('_instrument', '', regex=False)

        # Remove both '_fund' and '_instrument' suffixes from column names
        df_final.columns = df_final.columns.str.replace('_fund', '', regex=False).str.replace('_instrument', '', regex=False)


        # Save merged data
        df_final.to_excel(updated_csv_filename, index=False)
        print(f"Final data saved to {updated_csv_filename}.")
    else:
        print("No instrument data fetched.")


# Index Data ------------------------------------------------------------------------------------

def fetch_and_process_index_data(api_url, output_file):
    """
    Fetch index data from January 1, 2023, to today, filter correctly, 
    calculate daily returns, and save to an Excel file with sheet name 'Index'.
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    start_date = datetime(2025, 2, 17)  # Start date
    end_date = datetime.today()  # End date: Today

    all_index_data = []

    print("📢 Fetching index data. This may take some time...")

    # Fetch all available data
    response = requests.get(api_url, headers=headers)
    if response.status_code == 200:
        json_data = response.json()

        if 'indexB2' in json_data and json_data['indexB2']:
            df = pd.DataFrame(json_data['indexB2'])

            if 'xNivInuClMresIbs' in df.columns:
                df.rename(columns={'dEven': 'Date', 'xNivInuClMresIbs': 'IndexClosingPrice'}, inplace=True)
                df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d')

                # 🛑 Drop all rows before the start date
                df = df[df['Date'] >= start_date]

                # ✅ Keep only the last valid closing price per date
                df = df.sort_values(by=['Date']).groupby('Date').last().reset_index()

                # ✅ Calculate daily return
                df['IndexReturn'] = df['IndexClosingPrice'].pct_change()

                # 🛑 Drop the first row (NaN return)
                df.dropna(subset=['IndexReturn'], inplace=True)

                # ✅ Keep only the required columns
                df = df[['Date', 'IndexReturn']]

                # Append to the main list
                all_index_data.append(df)
        else:
            print("⚠️ Warning: 'indexB2' key missing or empty in API response.")
    else:
        print(f"❌ Failed to fetch index data. Status code: {response.status_code}")
        return

    if all_index_data:
        df_index_combined = pd.concat(all_index_data, ignore_index=True)

        # ✅ Save to an Excel file
        df_index_combined.to_excel(output_file, sheet_name='Index', index=False)
        print(f"✅ Index data with returns saved to {output_file}.")
    else:
        print("❌ No index data fetched.")

# Merge ---------------------------------------------------------------------------------------- 
def merge_fund_and_index_data(fund_file, index_file, output_file):
    """
    Merges the fund data with the index data on date, 
    keeping all fund rows and leaving IndexReturn empty if no index data exists for a date.
    """
    print(f"🔄 Merging {fund_file} and {index_file}...")
    
    # Load fund data
    df_fund = pd.read_excel(fund_file, engine="openpyxl")
    
    # Load index data
    df_index = pd.read_excel(index_file, sheet_name="Index", engine="openpyxl")

    # Convert dates to datetime format
    df_fund["date"] = pd.to_datetime(df_fund["date"])
    df_index["Date"] = pd.to_datetime(df_index["Date"])

    # Merge fund and index data on the date column (left join to keep all fund dates)
    merged_df = pd.merge(df_fund, df_index, left_on="date", right_on="Date", how="left")

    # Drop the extra Date column from index dataset
    merged_df.drop(columns=["Date"], inplace=True)

    # 🛑 **Ensure percentage values are divided by 100 in the final output**
    percentage_columns = [
        'dailyReturn', 'weeklyReturn', 'monthlyReturn', 
        'quarterlyReturn', 'sixMonthReturn', 'annualReturn', 'CumulativeReturn', 'fiveBest',
        'stock', 'bond', 'other', 'cash', 'deposit', 'insInvPercent', 'legalPercent', 'naturalPercent',
        'retInvPercent',  
    ]

    for col in percentage_columns:
        if col in merged_df.columns:
            merged_df[col] = merged_df[col] / 100

    # Save merged data
    merged_df.to_excel(output_file, index=False, engine="openpyxl")
    print(f"✅ Merged data saved to {output_file}")


In [2]:
import os
import pandas as pd
from datetime import datetime
import openpyxl

def update_fund_data(file_path, index_api_url, output_index_file):
    """ Updates fund data by fetching new data from the last date in the Excel file, 
        processing it, merging it with index data, and appending it to the existing file. """
    
    # ✅ Step 1: Ensure the input file exists
    if not os.path.exists(file_path):
        print(f"❌ Error: The file '{file_path}' does not exist. Please check the path.")
        return

    # Read previous fund data and rename for clarity.
    df_previous = pd.read_excel(file_path, engine='openpyxl')
    df_previous['date'] = pd.to_datetime(df_previous['date'])
    last_date = df_previous['date'].max()
    print(f"📅 Last date in the file: {last_date.strftime('%Y-%m-%d')}")

    # ✅ Step 2: Define the update date range
    start_date = last_date  # Start from the last available date
    end_date = datetime.today()  # Up to today
    print(f"🔄 Fetching new data from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}...")

    # ✅ Step 3: Temporary filenames
    csv_filename = "fund_data_temp.csv"
    updated_csv_filename = "updated_fund_data_temp.xlsx"
    output_filename = "instrument_data_temp.xlsx"
    final_output_file = "final_fund_index_data.xlsx"

    # ✅ Step 4: Fetch new fund data
    print("📡 Fetching and processing fund data...")
    raw_data = fetch_and_process_data(start_date, end_date, csv_filename)
    if raw_data.empty:
        print("⚠️ No new data was fetched. Please check the API or data source.")
        return

    # ✅ Step 5: Read the new CSV and extract required information
    if os.path.exists(csv_filename):
        print(f"📖 Reading new data from {csv_filename}...")
        df_temp = pd.read_csv(csv_filename)
        regnos = df_temp['regNo'].drop_duplicates().tolist() if 'regNo' in df_temp.columns else []
        dates = df_temp['date'].drop_duplicates().tolist() if 'date' in df_temp.columns else []

        # ✅ Step 6: Fetch and merge instrument data
        if regnos and dates:
            print("🔗 Fetching instrument data...")
            merge_instrument_data(regnos, dates, output_filename, csv_filename, updated_csv_filename)

        # ✅ Step 7: Process mutual fund licenses and post-process data
        if os.path.exists(updated_csv_filename):
            print("⚙️ Processing mutual fund licenses...")
            process_mutual_fund_licenses(updated_csv_filename)

            print("🛠️ Post-processing fund data...")
            post_process_data(updated_csv_filename)
            

            # ✅ Step 8: Fetch and process index data
            print("📊 Fetching and processing index data...")
            fetch_and_process_index_data(index_api_url, output_index_file)

            print(f"📝 Merging fund and index data into {final_output_file}...")
            merge_fund_and_index_data(updated_csv_filename, output_index_file, final_output_file)

            # ✅ Step 9: Read merged fund/index file for final concatenation
            df_new_merged = pd.read_excel(final_output_file, engine='openpyxl')
            df_new_merged['date'] = pd.to_datetime(df_new_merged['date'])

            # ✅ Step 10: Concatenate previous and new merged fund/index data
            df_final = pd.concat([df_previous, df_new_merged]).drop_duplicates(subset=['date', 'regNo'], keep='last')

            # ✅ Step 11: Save the final merged file
            with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
                df_final.to_excel(writer, index=False, sheet_name="fund_panel_data")
            print(f"✅ Final updated file saved as: {file_path} (Sheet: 'fund_panel_data')")

        else:
            print("❌ Error: Updated fund data file does not exist after processing. Please check previous steps.")
            return
    else:
        print("❌ Error: CSV file does not exist. Please check previous steps.")
        return

    # ✅ Step 12: Cleanup intermediate files
    print("🗑️ Cleaning up intermediate files...")
    files_to_delete = [csv_filename, updated_csv_filename, output_filename, output_index_file]
    if os.path.exists(final_output_file):
        print(f"✅ Process completed successfully. Final merged file saved as: {final_output_file}")
        for file in files_to_delete:
            if os.path.exists(file):
                os.remove(file)
                print(f"🗑️ Deleted: {file}")
        print("✅ Cleanup complete.")
    else:
        print("❌ ERROR: Final output file was NOT saved! Skipping cleanup to prevent data loss.")

# -----------------------------------------
# ✅ Main Execution Flow (Runs Every Day)
# -----------------------------------------
if __name__ == "__main__":
    file_path = "fund_data_latest.xlsx"
    index_api_url = "https://cdn.tsetmc.com/api/Index/GetIndexB2History/32097828799138957"
    output_index_file = "index_data.xlsx"
    update_fund_data(file_path, index_api_url, output_index_file)


📅 Last date in the file: 2025-08-04
🔄 Fetching new data from 2025-08-04 to 2025-09-02...
📡 Fetching and processing fund data...
Fund types fetched successfully.
Fetching data from API for date: 2025-08-04
Data fetched successfully for 2025-08-04
Fetching data from API for date: 2025-08-05
Data fetched successfully for 2025-08-05
Fetching data from API for date: 2025-08-06
Data fetched successfully for 2025-08-06
Fetching data from API for date: 2025-08-07
Data fetched successfully for 2025-08-07
Fetching data from API for date: 2025-08-08
Data fetched successfully for 2025-08-08
Fetching data from API for date: 2025-08-09
Data fetched successfully for 2025-08-09
Fetching data from API for date: 2025-08-10
Data fetched successfully for 2025-08-10
Fetching data from API for date: 2025-08-11
Data fetched successfully for 2025-08-11
Fetching data from API for date: 2025-08-12
Data fetched successfully for 2025-08-12
Fetching data from API for date: 2025-08-13
Data fetched successfully for 

C:\Users\a.shokatabadi.TSD0\AppData\Local\Temp\ipykernel_12992\3610548991.py:266: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_instruments = pd.concat(all_results, ignore_index=True)


Instrument data saved to instrument_data_temp.xlsx.
Final data saved to updated_fund_data_temp.xlsx.
⚙️ Processing mutual fund licenses...
Data successfully processed and saved to updated_fund_data_temp.xlsx with sheet name 'Fund_data'.
🛠️ Post-processing fund data...
Reading post-processing data from updated_fund_data_temp.xlsx...


C:\Users\a.shokatabadi.TSD0\AppData\Local\Temp\ipykernel_12992\3610548991.py:141: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(r'[\u200c\u00A0ـ]', ' ', str(x)) if isinstance(x, str) else x)
C:\Users\a.shokatabadi.TSD0\AppData\Local\Temp\ipykernel_12992\3610548991.py:188: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(r'نو\s+ویرا', 'نوویرا', str(x)) if isinstance(x, str) else x)
C:\Users\a.shokatabadi.TSD0\AppData\Local\Temp\ipykernel_12992\3610548991.py:196: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(r'سبد\s+گردان', 'سبدگردان', str(x)) if isinstance(x, str) else x)
C:\Users\a.shokatabadi.TSD0\AppData\Local\Temp\ipykernel_12992\3610548991.py:199: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(r'سرمایه\

Post-processing completed. Updated data saved to updated_fund_data_temp.xlsx.
📊 Fetching and processing index data...
📢 Fetching index data. This may take some time...
✅ Index data with returns saved to index_data.xlsx.
📝 Merging fund and index data into final_fund_index_data.xlsx...
🔄 Merging updated_fund_data_temp.xlsx and index_data.xlsx...
✅ Merged data saved to final_fund_index_data.xlsx
✅ Final updated file saved as: fund_data_latest.xlsx (Sheet: 'fund_panel_data')
🗑️ Cleaning up intermediate files...
✅ Process completed successfully. Final merged file saved as: final_fund_index_data.xlsx
🗑️ Deleted: fund_data_temp.csv
🗑️ Deleted: updated_fund_data_temp.xlsx
🗑️ Deleted: instrument_data_temp.xlsx
🗑️ Deleted: index_data.xlsx
✅ Cleanup complete.


# Tse Update

In [3]:
import pytse_client as tse
import datetime
import pandas as pd
import time
from requests.exceptions import ConnectionError, Timeout
import os

def update_tse_data(csv_file="all_tse_data.csv", inscode_file="inscode.xlsx"):
    # Check if the main CSV file exists
    if not os.path.exists(csv_file):
        print(f"Error: {csv_file} not found. Please run the initial download code first.")
        return
    
    # Read existing data
    existing_data = pd.read_csv(csv_file)
    existing_data['date'] = pd.to_datetime(existing_data['date'])
    
    # Get the latest date in the existing data
    latest_date = existing_data['date'].max()
    print(f"Latest date in existing data: {latest_date.date()}")
    
    # Read the Excel file for symbols
    inscode_df = pd.read_excel(inscode_file, sheet_name="Sheet2")
    inscode_df = inscode_df.dropna(subset="smallSymbolName")
    
    # Create a dictionary to map symbols to their regNo
    symbol_regno_map = dict(zip(inscode_df['smallSymbolName'], inscode_df['regNo']))
    
    # Get the ticker list
    ticker = inscode_df["smallSymbolName"].astype(str)
    
    # Create an empty list to store new data
    new_data = []
    
    # Process each ticker with error handling
    for t in ticker:
        try:
            print(f"Downloading updates for {t}...")
            ticker_data = tse.download(symbols=t, write_to_csv=False)
            
            # If ticker_data is a dictionary (multiple symbols), convert to list of dataframes
            if isinstance(ticker_data, dict):
                dfs = list(ticker_data.values())
            else:
                dfs = [ticker_data]
            
            # Process each dataframe
            for df in dfs:
                # Convert date column to datetime
                df['date'] = pd.to_datetime(df['date'])
                
                # Filter only new data
                new_records = df[df['date'] > latest_date].copy()
                
                if not new_records.empty:
                    new_records['symbol'] = t
                    new_records['regNo'] = symbol_regno_map.get(t)
                    new_data.append(new_records)
                    print(f"Found {len(new_records)} new records for {t}")
            
            time.sleep(2)  # Small delay between requests
            
        except (ConnectionError, Timeout) as e:
            print(f"Error downloading {t}: {str(e)}")
            print("Waiting 30 seconds before continuing...")
            time.sleep(30)
        except Exception as e:
            print(f"Unexpected error for {t}: {str(e)}")
    
    # If we have new data, append it to the existing file
    if new_data:
        try:
            new_df = pd.concat(new_data, ignore_index=True)
            print(f"Total new records found: {len(new_df)}")
            
            # Combine existing and new data
            updated_df = pd.concat([existing_data, new_df], ignore_index=True)
            
            # Sort by date and symbol
            updated_df = updated_df.sort_values(['date', 'symbol'])
            
            # Save the updated data
            updated_df.to_csv(csv_file, index=False)
            print(f"Successfully updated {csv_file}")
            print(f"Total records after update: {len(updated_df)}")
            
        except Exception as e:
            print(f"Error saving updated data: {str(e)}")
    else:
        print("No new data found to update")

if __name__ == "__main__":
    # Create a log file for the current run
    log_file = f"update_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    
    # Redirect stdout to both console and log file
    import sys
    class Logger:
        def __init__(self, filename):
            self.terminal = sys.stdout
            self.log = open(filename, "w", encoding='utf-8')
        
        def write(self, message):
            self.terminal.write(message)
            self.log.write(message)
            self.log.flush()
            
        def flush(self):
            self.terminal.flush()
            self.log.flush()
    
    sys.stdout = Logger(log_file)
    
    # Run the update
    print(f"Starting update at {datetime.datetime.now()}")
    update_tse_data()
    print(f"Update completed at {datetime.datetime.now()}")

Starting update at 2025-09-02 12:46:13.673733
Latest date in existing data: 2025-08-04
Found 18 new records for گنج
Found 18 new records for گلدیس
Found 17 new records for عقیق
Found 18 new records for تابش
Found 13 new records for دی سهام
Found 7 new records for پناه
Found 15 new records for مانا
Found 18 new records for آتیه ملت
Found 18 new records for ارمغان
Found 18 new records for صنهال
Found 18 new records for اصیل
Found 18 new records for آسام
Found 18 new records for هیبرید
Found 18 new records for کاریس
Found 11 new records for بذر
Found 18 new records for آسان
Found 16 new records for صنوین
Found 14 new records for الماس
Found 18 new records for ستاره
Found 18 new records for اطلس
Found 18 new records for فیروزه
Found 18 new records for تداوم
Found 18 new records for کاردان
Found 18 new records for اعتماد
Found 17 new records for خبرگان
Found 18 new records for صایند
Found 18 new records for سخند
Found 18 new records for ثروتم
Found 18 new records for آگاس
Unexpected error f

c:\Users\a.shokatabadi.TSD0\AppData\Local\anaconda3\Lib\site-packages\pytse_client\download.py:119: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(


Found 18 new records for امین یکم
Found 18 new records for پادا
Found 18 new records for رشدی کیان
Found 18 new records for گنجین
Found 18 new records for کارین
Found 18 new records for طلا
Found 18 new records for کمند
Found 18 new records for زر
Found 18 new records for گوهر
Found 18 new records for فیروزا
Found 17 new records for افق ملت
Found 18 new records for عیار
Found 18 new records for تصمیم
Found 18 new records for اوصتا
Found 18 new records for آسامید
Found 18 new records for سرو
Found 18 new records for دارا
Found 18 new records for گنجینه
Found 2 new records for ونچر
Found 18 new records for افران
Found 18 new records for یاقوت
Found 18 new records for دارا یکم
Found 18 new records for ارزش
Found 18 new records for داریک
Found 18 new records for آوا
Found 18 new records for سپر
Found 18 new records for مدیر
Found 18 new records for پالایش
Found 18 new records for خاتم
Found 18 new records for نهال
Found 18 new records for سحرخیز
Found 18 new records for زرین
Found 18 new r